### **Task 3: Build a Chatbot using Hugging Face Transformers**

In [1]:
!pip install transformers torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)

In [4]:
def get_rule_based_answer(user_input):
    text = user_input.lower()

    if "hello" in text:
        return "Hello! Nice to meet you. How can I assist you today?"

    elif "what is artificial intelligence" in text or "what is ai" in text:
        return "Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving."

    elif "who created python" in text:
        return "Python was created by Guido van Rossum and released in 1991."

    elif "thank you" in text:
        return "You're welcome! Feel free to ask more questions."

    return None

In [5]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to end.\n")

chat_history_ids = None

while True:
    user_input = input("You: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day 😊")
        break

    # Rule-based answers (for accuracy)
    rule_response = get_rule_based_answer(user_input)
    if rule_response:
        print(f"Chatbot: {rule_response}\n")
        continue

    # Tokenize input
    new_input = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt')
    new_input_ids = new_input['input_ids']
    attention_mask = new_input['attention_mask']

    # Maintain chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=500,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.7,
        no_repeat_ngram_size=3
    )

    # Decode response
    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print(f"Chatbot: {bot_response}\n")

Chatbot: Hello! I am your AI assistant. Type 'exit' to end.



You:  Hello


Chatbot: Hello! Nice to meet you. How can I assist you today?



You:  What is Artificial Intelligence?


Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.



...

Chatbot: Python was created by Guido van Rossum and released in 1991.



You:  Thank you


Chatbot: You're welcome! Feel free to ask more questions.



You:  exit


Chatbot: Goodbye! Have a great day 😊
